# TODO:

    1. Research which loss function to use
    2. Research which optimization function to use
    3. Research which model will be best to use

In [1]:
from pathlib import Path
import polars as pl

In [2]:
data_path = '../data/'

is_training_data = True

if is_training_data:
    saving_path = data_path + 'Final Training Data/'
    working_data_path = data_path + 'Training Data/'
else:
    saving_path = data_path + 'Final Data For Labeling/'
    working_data_path = data_path + 'Data For Labeling/'

In [3]:
gaze_CSV_path = saving_path + 'gaze.csv'
fixations_CSV_path = saving_path + 'fixations.csv'
saccades_CSV_path = saving_path + 'saccades.csv'
imu_CSV_path = saving_path + 'imu.csv'
blinks_CSV_path = saving_path + 'blinks.csv'
eye_3D_states_CSV_path = saving_path + '3d_eye_states.csv'
label_CSV_path = saving_path + 'labels.csv'

I decided to create one CSV file, for each feature, that contains all the rows for each participant.

In [4]:
def merge_csvs(root_dir, target_filename, output_file):
    root = Path(root_dir)
    # List that holds each found CSV file.
    all_dfs = []
    
    # Iterate over the root dir.
    for subdir in root.iterdir():
        # If there is a subdir open it and try to find the file we are searching for in it.
        if subdir.is_dir():
            csv_path = subdir / target_filename

            # If we find it, read it and append it to the list with found CSVs.
            if csv_path.exists():
                df = pl.scan_csv(csv_path, schema_overrides={
                    "fixation id": pl.Utf8,
                    "blink id": pl.Utf8,
                    "saccades id": pl.Utf8,
                })
                all_dfs.append(df)

    # If there are no CSVs found.
    if not all_dfs:
        raise ValueError("No CSV files found.")

    # Merge all found CSVs.
    merged_df = pl.concat(all_dfs, how="vertical")
    # Create new CSV file with all rows of all found CSVs 
    merged_df.sink_csv(output_file)

    return merged_df

In [5]:
gaze_DF = merge_csvs(working_data_path, 'gaze.csv', gaze_CSV_path)
fixation_DF = merge_csvs(working_data_path, 'fixations.csv', fixations_CSV_path)
saccades_DF = merge_csvs(working_data_path, 'saccades.csv', saccades_CSV_path)
imu_DF = merge_csvs(working_data_path, 'imu.csv', imu_CSV_path)
blinks_DF = merge_csvs(working_data_path, 'blinks.csv', blinks_CSV_path)
eye_3D_states_DF = merge_csvs(working_data_path, '3d_eye_states.csv', eye_3D_states_CSV_path)

In [6]:
if is_training_data:
    labels_DF = merge_csvs(working_data_path, "labels.csv", label_CSV_path)
    # Sort (required before shift logic)
    labels_DF = labels_DF.sort([
        "recording id",
        "start timestamp [ms]"
    ])

    labels_DF = labels_DF.with_columns([
        # Previous row values (equivalent to shift)
        pl.col("label").shift(1).alias("_prev_label"),
        pl.col("recording id").shift(1).alias("_prev_recording"),
        pl.col("end timestamp [ms]").shift(1).alias("_prev_end")
    ])

    # Create grouping condition
    labels_DF = labels_DF.with_columns([
        (
            (pl.col("label") != pl.col("_prev_label")) |
            (pl.col("recording id") != pl.col("_prev_recording")) |
            (pl.col("start timestamp [ms]") != pl.col("_prev_end"))
        ).cast(pl.Int64)
        .cum_sum()
        .alias("group")
    ])

    # Aggregate per group
    labels_DF = labels_DF.group_by("group").agg([
        pl.col("recording id").first(),
        pl.col("start timestamp [ms]").first(),
        pl.col("end timestamp [ms]").last(),
        pl.col("label").first()
    ])

    # Remove helper column
    labels_DF = labels_DF.drop("group")

    # Save
    labels_DF.sink_csv(label_CSV_path)


First I will combine most of the data into one CSV file and train a model with it. Then I will remove some data that does not feel relavent and see what works best. After I find the best comination of type of that I will try to extract more data out of the one i already have. For example insted of leaving the rows where the perticipant have blinked, I can remove it and calc the blinking rate for given frame. direc

I map the saccade Id onto the gaze data

In [7]:
saccades_DF = saccades_DF.drop(["section id"])

gaze_DF = gaze_DF.sort(["recording id", "timestamp [ns]"])
saccades_DF = saccades_DF.sort(["recording id", "start timestamp [ns]"])

gaze_DF = gaze_DF.join_asof(
    saccades_DF,
    left_on="timestamp [ns]",
    right_on="start timestamp [ns]",
    by="recording id",
    strategy="backward",
).filter(
    pl.col("timestamp [ns]") <= pl.col("end timestamp [ns]")
)


In [8]:
def build_labeled_gaze_streaming(gaze_LF: pl.LazyFrame, labels_LF: pl.LazyFrame):

    # ----------------------------
    # 1. Pre-sort once (important)
    # ----------------------------
    gaze_LF = gaze_LF.sort(["recording id", "timestamp [ns]"])
    labels_LF = labels_LF.sort(["recording id", "start timestamp [ms]"])

    # ----------------------------
    # 2. Process per recording (NO global join)
    # ----------------------------
    def process_recording(recording_id):

        gaze = gaze_LF.filter(pl.col("recording id") == recording_id)
        labels = labels_LF.filter(pl.col("recording id") == recording_id)

        gaze = gaze.with_columns(
            ((pl.col("timestamp [ns]") - pl.col("timestamp [ns]").min()) / 1e6).alias("time_ms")
        )

        # small in-memory join per subject (safe)
        joined = (
            gaze.join(labels, how="cross")
            .filter(
                (pl.col("time_ms") >= pl.col("start timestamp [ms]")) &
                (pl.col("time_ms") <= pl.col("end timestamp [ms]"))
            )
            .group_by("time_ms")
            .agg(pl.col("label").first())
        )

        return gaze.join(joined, on="time_ms", how="left")

    # ----------------------------
    # 3. Stream execution over recordings
    # ----------------------------
    recordings = gaze_LF.select("recording id").unique().collect().to_series()

    result = []
    for r in recordings:
        result.append(process_recording(r))

    return pl.concat(result)


if is_training_data: 
    gaze_DF = build_labeled_gaze_streaming(gaze_DF, labels_DF)

/tmp/ipykernel_205/4274509474.py:37: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  recordings = gaze_LF.select("recording id").unique().collect().to_series()


Now I am combining all data into one CSV file. Firstly, I normalize the timestemps just to be sure they are in the same format as I will use them for sinhronization. I am using the gaze data as the base as it was capchured in 200 Hz and so it will be best if all other data is mapped to it. After normilizing the timestemps, I am mergeing the other files with the gaze data based on the timestamp or the Id of the event.

In [9]:
# Normalize timestamps
def normalize_timestamps(*dfs):
    out = []

    for df in dfs:
        exprs = []
        for col in df.collect_schema().names():
            if "timestamp" in col:
                exprs.append(
                    pl.col(col).cast(pl.Int64, strict=False)
                )

        if exprs:
            df = df.with_columns(exprs)

        out.append(df)

    return out


def merge_event(base, other, on, included_cols, suffix):
    other = other.select(included_cols)

    return base.join(
        other,
        on=on,
        how="left",
        suffix=suffix
    )

# Merge continuous signals
def merge_continuous_data(base, other, ts_col, suffix, tolerance_ns=50_000_000):
    other = other.sort(ts_col)
    base = base.sort(ts_col)

    return base.join_asof(
        other,
        on=ts_col,
        strategy="nearest",
        suffix=suffix
    )

def merge_all():
    # Normalize timestamps
    gaze, fixation, saccades, blinks, imu, eye_3D_states = normalize_timestamps(
        gaze_DF, fixation_DF, saccades_DF, blinks_DF, imu_DF, eye_3D_states_DF
    )
    
    fix_cols = [
        "fixation id",
        "duration [ms]",
        "fixation x [px]",
        "fixation y [px]"
    ]

    sacc_cols = [
        "saccade id",
        "duration [ms]",
        "amplitude [deg]",
        "mean velocity [px/s]",
        "peak velocity [px/s]"
    ]

    blink_cols = [
        "blink id",
        "duration [ms]"
    ]

    
    # Map interval events
    gaze = merge_event(gaze, fixation, "fixation id", fix_cols, " fix")
    gaze = merge_event(gaze, saccades, "saccade id", sacc_cols, " sacc")
    gaze = merge_event(gaze, blinks, "blink id", blink_cols, " blink")
    
    # Merge IMU
    gaze = merge_continuous_data(
        gaze, imu_DF,
        "timestamp [ns]", " imu"
    )

    # Merge Eye 3D
    gaze = merge_continuous_data(
        gaze, eye_3D_states_DF,
        "timestamp [ns]"," eye3d"
    )

    return gaze


merged_df = merge_all()

I am removing the rows where the "worn" column, in the gaze.csv, is equal to 0. Then I am dropping the "worn" column as it does not contain any useful information anymore

In [10]:
merged_df = merged_df.filter(
    pl.col("worn") != 0
).drop([
    "worn",
    "section id imu",
    "recording id imu",
    "section id eye3d",
    "recording id eye3d",
    "gaze mono left x [px]",
    "gaze mono left y [px]",
    "gaze mono right x [px]",
    "gaze mono right y [px]"
])

In [11]:
merged_df = merged_df.with_columns(
    ((pl.col("timestamp [ns]") - pl.col("timestamp [ns]").min().over("recording id")) / 1e9).alias("time sec")
)

merged_df = merged_df.with_columns(
    pl.col("fixation id").cast(pl.Int64).fill_null(-1), 
    pl.col("blink id").cast(pl.Int64).fill_null(-1)
)

merged_df = merged_df.with_columns(
    pl.col("duration [ms] blink").fill_null(-1), 
    pl.col("fixation y [px]").fill_null(-1),
    pl.col("fixation x [px]").fill_null(-1), 
    pl.col("duration [ms] fix").fill_null(-1)
).collect()


/tmp/ipykernel_205/3079758634.py:15: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  ).collect()


In [12]:
merged_df.write_parquet(saving_path + "merged_output.parquet")
print("Parquet saved as merged_output.parquet")
merged_df.write_csv(saving_path + "merged_output.csv")
print("Merged dataset saved as merged_output.csv")

Parquet saved as merged_output.parquet
Merged dataset saved as merged_output.csv


In [13]:
merged_df["label"].unique()

label
str
"""Searching for pieces"""
"""Undefined"""
"""Assembling"""
"""Looking at guide book"""


In [14]:
df = merged_df.group_by("recording id").agg(
    pl.col("label").unique().alias("labels")
)

for rec_id, labels in df.iter_rows():
    print(f"\nRecording: {rec_id}")
    print(labels)


Recording: 63efffc7-e347-47fb-9615-f7da183d8792
['Looking at guide book', 'Searching for pieces', 'Assembling', 'Undefined']

Recording: 63d66e1c-434f-4f23-8e74-abd8dedc0e43
['Looking at guide book', 'Assembling', 'Undefined', 'Searching for pieces']

Recording: e2e75728-4988-49c4-b2cc-9ec7a8bd96bc
['Looking at guide book', 'Searching for pieces', 'Assembling', 'Undefined']

Recording: 911806b6-27bd-4b56-bd2f-45d979842721
['Looking at guide book', 'Searching for pieces', 'Assembling', 'Undefined']

Recording: f79e6fc3-2075-4b72-b8d6-473f2d2e6694
['Searching for pieces', 'Looking at guide book', 'Assembling', 'Undefined']

Recording: 853a8f80-6e9a-4e3b-9312-522e2ec6f822
['Looking at guide book', 'Searching for pieces', 'Undefined', 'Assembling']

Recording: b636a895-09a2-4fe2-9c37-973ed9687a60
['Searching for pieces', 'Looking at guide book', 'Assembling', 'Undefined']

Recording: 1279952d-14d4-4e77-9010-a000dd546bcd
['Looking at guide book', 'Searching for pieces', 'Assembling', 'Unde

In [15]:
merged_df.filter(
    pl.col("label").is_null()
)

section id,recording id,timestamp [ns],gaze x [px],gaze y [px],fixation id,blink id,azimuth [deg],elevation [deg],saccade id,start timestamp [ns],end timestamp [ns],duration [ms],amplitude [px],amplitude [deg],mean velocity [px/s],peak velocity [px/s],time_ms,label,duration [ms] fix,fixation x [px],fixation y [px],duration [ms] sacc,amplitude [deg] sacc,mean velocity [px/s] sacc,peak velocity [px/s] sacc,duration [ms] blink,gyro x [deg/s],gyro y [deg/s],gyro z [deg/s],acceleration x [g],acceleration y [g],acceleration z [g],roll [deg],pitch [deg],yaw [deg],quaternion w,quaternion x,quaternion y,quaternion z,pupil diameter left [mm],pupil diameter right [mm],eyeball center left x [mm],eyeball center left y [mm],eyeball center left z [mm],eyeball center right x [mm],eyeball center right y [mm],eyeball center right z [mm],optical axis left x,optical axis left y,optical axis left z,optical axis right x,optical axis right y,optical axis right z,eyelid angle top left [rad],eyelid angle bottom left [rad],eyelid aperture left [mm],eyelid angle top right [rad],eyelid angle bottom right [rad],eyelid aperture right [mm],time sec
str,str,i64,f64,f64,i64,i64,f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,str,i64,f64,f64,i64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64


In [16]:
merged_df.filter(
    pl.any_horizontal(pl.all().is_null())
)

section id,recording id,timestamp [ns],gaze x [px],gaze y [px],fixation id,blink id,azimuth [deg],elevation [deg],saccade id,start timestamp [ns],end timestamp [ns],duration [ms],amplitude [px],amplitude [deg],mean velocity [px/s],peak velocity [px/s],time_ms,label,duration [ms] fix,fixation x [px],fixation y [px],duration [ms] sacc,amplitude [deg] sacc,mean velocity [px/s] sacc,peak velocity [px/s] sacc,duration [ms] blink,gyro x [deg/s],gyro y [deg/s],gyro z [deg/s],acceleration x [g],acceleration y [g],acceleration z [g],roll [deg],pitch [deg],yaw [deg],quaternion w,quaternion x,quaternion y,quaternion z,pupil diameter left [mm],pupil diameter right [mm],eyeball center left x [mm],eyeball center left y [mm],eyeball center left z [mm],eyeball center right x [mm],eyeball center right y [mm],eyeball center right z [mm],optical axis left x,optical axis left y,optical axis left z,optical axis right x,optical axis right y,optical axis right z,eyelid angle top left [rad],eyelid angle bottom left [rad],eyelid aperture left [mm],eyelid angle top right [rad],eyelid angle bottom right [rad],eyelid aperture right [mm],time sec
str,str,i64,f64,f64,i64,i64,f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,str,i64,f64,f64,i64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64


In [17]:
merged_df.select(
    pl.col("label").is_null().sum()
)

label
u32
0


In [18]:
labels_DF.filter(
    pl.col("start timestamp [ms]") == pl.col("end timestamp [ms]")
).collect()

recording id,start timestamp [ms],end timestamp [ms],label
str,i64,i64,str
"""911806b6-27bd-4b56-bd2f-45d979…",266587,266587,"""Searching for pieces"""


In [19]:
labels_DF.filter(
    pl.col("start timestamp [ms]") > pl.col("end timestamp [ms]")
).collect()

recording id,start timestamp [ms],end timestamp [ms],label
str,i64,i64,str
